## Read Data

In [8]:
import pandas as pd
import matplotlib.pyplot as plt
import glob
import os

def read_tec_data(file_path):
    """Read TEC data file and return a pandas DataFrame."""
    columns = ['Week', 'GPS_TOW', 'PRN', 'Az', 'Elv', 'L1CN0', 'S4', 'S4Cor', 
              'TEC30', 'TECRate30', 'L1LockTime', 'L2LockTime', 'L2CN0']
    
    df = pd.read_csv(file_path, names=columns, delimiter=',')
    
    # Convert relevant columns to numeric (you can add more if needed)
    for col in ['TEC30', 'TECRate30', 'GPS_TOW', 'Az', 'Elv', 'S4']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    return df

def plot_tec_data(data, station_id):
    """Create plots for TEC data."""
    plt.figure(figsize=(15, 10))
    
    # Plot 1: TEC values over time for each PRN
    plt.subplot(2, 2, 1)
    for prn in data['PRN'].unique():
        prn_data = data[data['PRN'] == prn]
        plt.plot(prn_data['GPS_TOW'], prn_data['TEC30'], label=f'PRN {prn}')
    plt.title(f'TEC Values Over Time - Station {station_id}')
    plt.xlabel('GPS Time of Week (seconds)')
    plt.ylabel('TEC (TECU)')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    
    # Plot 2: S4 Scintillation Index
    plt.subplot(2, 2, 2)
    for prn in data['PRN'].unique():
        prn_data = data[data['PRN'] == prn]
        plt.plot(prn_data['GPS_TOW'], prn_data['S4'], label=f'PRN {prn}')
    plt.title(f'S4 Scintillation Index - Station {station_id}')
    plt.xlabel('GPS Time of Week (seconds)')
    plt.ylabel('S4 Index')
    
    # Plot 3: Elevation vs Azimuth (Sky Plot)
    plt.subplot(2, 2, 3)
    plt.scatter(data['Az'], data['Elv'], c=data['TEC30'], cmap='viridis')
    plt.title(f'Satellite Sky Plot - Station {station_id}')
    plt.xlabel('Azimuth (degrees)')
    plt.ylabel('Elevation (degrees)')
    plt.colorbar(label='TEC')
    
    # Plot 4: TEC Rate
    plt.subplot(2, 2, 4)
    for prn in data['PRN'].unique():
        prn_data = data[data['PRN'] == prn]
        plt.plot(prn_data['GPS_TOW'], prn_data['TECRate30'], label=f'PRN {prn}')
    plt.title(f'TEC Rate - Station {station_id}')
    plt.xlabel('GPS Time of Week (seconds)')
    plt.ylabel('TEC Rate (TECU/min)')
    
    plt.tight_layout()
    return plt.gcf()

# Path to the data directory
data_dir = "../Data/TEC_2D_Data/One_Day_Data"

# Get all .dat files
data_files = glob.glob(os.path.join(data_dir, "*.dat"))

for file_path in data_files:
    # Extract station ID from filename
    station_id = os.path.basename(file_path).split('_')[0]
    
    # Read data
    print(f"Processing station {station_id}...")
    data = read_tec_data(file_path)
    
    # Create plots
    fig = plot_tec_data(data, station_id)
    
    # Save the figure
    output_dir = "../Output"
    os.makedirs(output_dir, exist_ok=True)
    fig.savefig(os.path.join(output_dir, f"tec_analysis_station_{station_id}.png"))
    plt.close(fig)
    
    # Print basic statistics
    print(f"\nStation {station_id} Statistics:")
    print(f"Number of satellites: {len(data['PRN'].unique())}")
    print(f"Time span: {data['GPS_TOW'].max() - data['GPS_TOW'].min()} seconds")
    print(f"Average TEC: {data['TEC30'].mean():.2f} TECU")
    print(f"Max TEC: {data['TEC30'].max():.2f} TECU")
    print(f"Min TEC: {data['TEC30'].min():.2f} TECU")
    print("-" * 50)


Processing station 124...

Station 124 Statistics:
Number of satellites: 29
Time span: 86400.0 seconds
Average TEC: 33.91 TECU
Max TEC: 139.99 TECU
Min TEC: 0.00 TECU
--------------------------------------------------
Processing station 212...

Station 212 Statistics:
Number of satellites: 29
Time span: 86400.0 seconds
Average TEC: 32.39 TECU
Max TEC: 141.03 TECU
Min TEC: 0.00 TECU
--------------------------------------------------
Processing station 213...

Station 213 Statistics:
Number of satellites: 29
Time span: 86400.0 seconds
Average TEC: 32.18 TECU
Max TEC: 117.15 TECU
Min TEC: -4.22 TECU
--------------------------------------------------
Processing station 214...

Station 214 Statistics:
Number of satellites: 30
Time span: 86400.0 seconds
Average TEC: 31.82 TECU
Max TEC: 156.96 TECU
Min TEC: -9.90 TECU
--------------------------------------------------
Processing station 301...

Station 301 Statistics:
Number of satellites: 29
Time span: 86400.0 seconds
Average TEC: 52.32 TECU

## Extracting Lat, Lon, Eca 

In [9]:
import pandas as pd
import numpy as np
import glob, os

def read_tec_data(file_path):
    columns = ['Week', 'GPS_TOW', 'PRN', 'Az', 'Elv', 'L1CN0', 'S4', 'S4Cor', 
               'TEC30', 'TECRate30', 'L1LockTime', 'L2LockTime', 'L2CN0']
    df = pd.read_csv(file_path, names=columns, delimiter=',', skiprows=1)
    for col in ['Az', 'Elv', 'TEC30', 'TECRate30', 'GPS_TOW', 'S4']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

def compute_ipp(df, R_lat_deg, R_lon_deg, h_I=350, r_e=6371):
    # Convert degrees to radians
    R_lat = np.radians(R_lat_deg)
    R_lon = np.radians(R_lon_deg)
    el = np.radians(df['Elv'])
    az = np.radians(df['Az'])

    # ECA calculation
    term = (r_e * np.cos(el)) / (r_e + h_I)
    eca = (np.pi / 2) - el - np.arcsin(term)

    # IPP latitude
    IPP_lat = np.arcsin(np.sin(R_lat) * np.cos(eca) + 
                        np.cos(R_lat) * np.sin(eca) * np.cos(az))

    # IPP longitude
    delta_lon = np.arcsin(np.sin(eca) * np.sin(az)) / np.cos(IPP_lat)
    IPP_lon = R_lon + delta_lon

    # Convert radians to degrees
    df['ECA'] = np.degrees(eca)
    df['IPP_lat'] = np.degrees(IPP_lat)
    df['IPP_lon'] = np.degrees(IPP_lon)
    return df

# ---- MAIN SCRIPT ----
data_dir = "../Data/TEC_2D_Data/One_Day_Data"
output_dir = "../Data/GenData/Processed_TEC_with_IPP"
os.makedirs(output_dir, exist_ok=True)

data_files = glob.glob(os.path.join(data_dir, "*.dat"))

# 🔧 You must set the receiver's latitude and longitude for each station
station_coords = {
    "stationA": (23.2, 77.5),   # Example: Bhopal
    "stationB": (28.6, 77.2),   # Example: Delhi
    # Add more if needed
}

for file_path in data_files:
    station_id = os.path.basename(file_path).split('_')[0]
    print(f"Processing station {station_id}...")

    data = read_tec_data(file_path)

    # Get station lat/lon or default
    R_lat, R_lon = station_coords.get(station_id, (23.0, 78.0))  # default to MP center

    # Compute IPP and ECA
    data = compute_ipp(data, R_lat, R_lon)

    # Save to new directory
    new_file = os.path.join(output_dir, f"{station_id}_with_IPP.csv")
    data.to_csv(new_file, index=False)
    print(f"Saved processed file to: {new_file}")


Processing station 124...
Saved processed file to: ../Data/GenData/Processed_TEC_with_IPP\124_with_IPP.csv
Processing station 212...
Saved processed file to: ../Data/GenData/Processed_TEC_with_IPP\212_with_IPP.csv
Processing station 213...
Saved processed file to: ../Data/GenData/Processed_TEC_with_IPP\213_with_IPP.csv
Processing station 214...
Saved processed file to: ../Data/GenData/Processed_TEC_with_IPP\214_with_IPP.csv
Processing station 301...
Saved processed file to: ../Data/GenData/Processed_TEC_with_IPP\301_with_IPP.csv
Processing station 302...
Saved processed file to: ../Data/GenData/Processed_TEC_with_IPP\302_with_IPP.csv
Processing station 305...
Saved processed file to: ../Data/GenData/Processed_TEC_with_IPP\305_with_IPP.csv
Processing station 306...
Saved processed file to: ../Data/GenData/Processed_TEC_with_IPP\306_with_IPP.csv
Processing station 307...
Saved processed file to: ../Data/GenData/Processed_TEC_with_IPP\307_with_IPP.csv
Processing station 308...
Saved proce